In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

print("Lading imagage ")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

X = np.transpose(X, (0, 3, 1, 2))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)


In [ ]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
data_iter = iter(train_loader)
images_batch, labels_batch = next(data_iter)
print(f"Batch Images shape: {images_batch.shape}")
print(f"Batch Labels shape: {labels_batch.shape}")


In [ ]:

plt.figure(figsize=(10, 4))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(images_batch[i].permute(1, 2, 0))
    plt.title(f"Age: {int(labels_batch[i].item())}")
    plt.axis('off')
plt.show()

In [ ]:
import torch.nn as nn
import torch.optim as optim

class AgePredictorModel(nn.Module):
    def __init__(self):
        super(AgePredictorModel, self).__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(3 * 36 * 36, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)


In [ ]:
def train_step(model, loader, loss_fn, optimizer, device):
    model.train()
    train_loss = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)

        preds = model(X)
        loss = loss_fn(preds, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    return train_loss / len(loader)


In [ ]:
def val_step(model, loader, loss_fn, device):
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = model(X)
            loss = loss_fn(preds, y)
            val_loss += loss.item()
    return val_loss / len(loader)


In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AgePredictorModel().to(device)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:

train_losses = []
val_losses = []
epochs = 20

print(f"Training on {device}...")
for epoch in range(epochs):
    t_loss = train_step(model, train_loader, loss_fn, optimizer, device)
    v_loss = val_step(model, test_loader, loss_fn, device)

    train_losses.append(t_loss)
    val_losses.append(v_loss)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}")

In [ ]:
# Task 1: Write your code here: okay -__-
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs+1), train_losses, label='Train Loss')
plt.plot(range(1, epochs+1), val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here: okay -__-
model.eval()
with torch.no_grad():
    samples, actual_ages = next(iter(test_loader))
    samples, actual_ages = samples.to(device), actual_ages.to(device)
    predicted_ages = model(samples)

plt.figure(figsize=(12,6))
for i in range(5):
    plt.subplot(1, 5, i+1)
    img = samples[i].cpu().permute(1,2,0)
    plt.imshow(img)
    plt.title(f"Pred: {predicted_ages[i].item():.1f}\nActual: {actual_ages[i].item():.1f}")
    plt.axis('off')
plt.tight_layout()
plt.show()